# test for vorarlberg

In [19]:
import os

import pandas as pd
import numpy as np

In [37]:
data_path = "../data/extracted/"

state_name = "20241212-0624_gtfs_vmobil_2024_vora/"

# select day for calculation in format YYYYMMDD in 2024
selected_day = 20240530

In [22]:

stops_df = pd.read_csv(data_path + state_name + "stops.txt", 
                        quotechar='"',
                        sep=",")

print(stops_df.head())

           stop_id                    stop_name   stop_lat   stop_lon  \
0   at:47:1222:0:4       St. Anton a.A. Bahnhof  47.127468  10.266639   
1    at:47:1222:22                    Steig 2+3  47.127450  10.267232   
2  at:47:61099:0:1    St. Anton a. A. Kohlereck  47.122358  10.253820   
3  at:47:61099:0:2    St. Anton a. A. Kohlereck  47.122322  10.253703   
4  at:47:62209:0:1  St. Anton a. A. Stadle B197  47.122273  10.248646   

   zone_id  location_type parent_station level_id platform_code  
0   6455.0            NaN    Pat:47:1222  Level 0             1  
1      NaN            NaN    Pat:47:1222  Level 0           NaN  
2   6455.0            NaN   Pat:47:61099  Level 0             1  
3   6455.0            NaN   Pat:47:61099  Level 0             2  
4   6455.0            NaN   Pat:47:62209  Level 0             1  


In [30]:
stop_times_df = pd.read_csv(data_path + state_name + "stop_times.txt", sep=',', quotechar='"')

# remove from stop_times_df entries outside of the time window (6am-8pm)
stop_times_df = stop_times_df[stop_times_df['departure_time'].between('06:00:00', '20:00:00')]

print(stop_times_df.head())

                     trip_id arrival_time departure_time         stop_id  \
27  1.T0.12-820-E-j24-1.20.R     06:00:00       06:00:00  at:48:1232:0:2   
28  1.T0.12-820-E-j24-1.20.R     06:01:00       06:01:00  at:48:1233:0:2   
29  1.T0.12-820-E-j24-1.20.R     06:02:00       06:02:00  at:48:1228:0:2   
30  1.T0.12-820-E-j24-1.20.R     06:03:00       06:03:00  at:48:1229:0:2   
31  1.T0.12-820-E-j24-1.20.R     06:05:00       06:05:00  at:48:1235:0:2   

    stop_sequence  stop_headsign  pickup_type  drop_off_type  \
27             28            NaN            0              0   
28             29            NaN            0              0   
29             30            NaN            0              0   
30             31            NaN            0              0   
31             32            NaN            0              0   

    shape_dist_traveled  
27             19261.74  
28             20149.77  
29             20924.20  
30             21372.79  
31             22450.46  


In [47]:
trips_df = pd.read_csv(data_path + state_name + 'trips.txt', sep=',', quotechar='"')
print(trips_df.shape)

(20739, 8)


In [33]:
calendar_df = pd.read_csv(data_path + state_name + "calendar.txt", sep=",", quotechar='"')

#calendar_dates_df = pd.read_csv(data_path + state_name + "calendar_dates.txt", sep=",", quotechar='"')

print(calendar_df)

            service_id  monday  tuesday  wednesday  thursday  friday  \
0                   T0       1        1          1         1       1   
1                 T0#1       1        1          1         1       1   
2             T0+02o00       1        1          1         1       1   
3             T0+05310       0        0          0         0       1   
4             T0+05p00       1        1          1         1       1   
..                 ...     ...      ...        ...       ...     ...   
842   TA-90-11-j24+T21       1        1          1         1       1   
843   TA-90-11-j24+T22       0        0          0         0       0   
844   TA-90-11-j24+T24       0        0          0         0       0   
845   TA-90-11-j24+T35       1        1          1         1       0   
846  TA-90-11-j24+T35!       0        0          0         0       1   

     saturday  sunday  start_date  end_date  
0           0       0    20231210  20241214  
1           0       0    20240708  20240913

In [48]:
# remove all services entries in calendar_df that are not relevant for the specified day

calendar_filtered_df = calendar_df[(calendar_df['start_date'] <= selected_day) & (calendar_df['end_date'] >= selected_day)]
# TODO: check exceptions from calendar_dates_df
print(calendar_filtered_df.shape)

(813, 10)


In [46]:
# remove entries from trips_df that are not valid
trips_filtered_df = trips_df[trips_df['service_id'].isin(calendar_filtered_df['service_id'])]
print(trips_filtered_df.shape)

(19548, 8)


In [50]:
print(stop_times_df.shape)

# keep only valid trips from trips_filtered_df
stop_times_df = stop_times_df[stop_times_df['trip_id'].isin(trips_filtered_df['trip_id'])]

print(stop_times_df.shape)

(347880, 9)
(342300, 9)


In [56]:
df = stops_df.merge(stop_times_df, on='stop_id')
result = df.groupby("parent_station").size().reset_index(name="count")
result.rename(columns={"parent_station": "stop_id"}, inplace=True)
print(result.head())

        stop_id  count
0   Pat:47:1222    174
1  Pat:47:61099    174
2  Pat:47:62209     87
3  Pat:47:62504     87
4  Pat:47:64938     69


In [63]:
# merge result with stops parent stations
stops_final_df = stops_df.merge(result, on='stop_id', how='right')
print(stops_final_df.head())

        stop_id                      stop_name   stop_lat   stop_lon  zone_id  \
0   Pat:47:1222   St. Anton am Arlberg Bahnhof  47.127389  10.266782      NaN   
1  Pat:47:61099      St. Anton a. A. Kohlereck  47.122346  10.253766      NaN   
2  Pat:47:62209    St. Anton a. A. Stadle B197  47.122267  10.248583      NaN   
3  Pat:47:62504        St. Anton a. A. Brandli  47.124901  10.260108      NaN   
4  Pat:47:64938  St. Anton a. A. Terminal West  47.126527  10.263333      NaN   

   location_type parent_station level_id platform_code  count  
0            1.0            NaN      NaN           NaN    174  
1            1.0            NaN      NaN           NaN    174  
2            1.0            NaN      NaN           NaN     87  
3            1.0            NaN      NaN           NaN     87  
4            1.0            NaN      NaN           NaN     69  


In [64]:
# calculate the PTSQL

stops_final_df["interval"] = stops_final_df["count"].apply(lambda x: 840 / (x/2))
display(stops_final_df)

,stop_id,stop_name,stop_lat,stop_lon,zone_id,location_type,parent_station,level_id,platform_code,count,interval
0,Pat:47:1222,St. Anton am Arlberg Bahnhof,47.127389,10.266782,NaN,1.0,NaN,NaN,NaN,174,9.655172
1,Pat:47:61099,St. Anton a. A. Kohlereck,47.122346,10.253766,NaN,1.0,NaN,NaN,NaN,174,9.655172
2,Pat:47:62209,St. Anton a. A. Stadle B197,47.122267,10.248583,NaN,1.0,NaN,NaN,NaN,87,19.310345
3,Pat:47:62504,St. Anton a. A. Brandli,47.124901,10.260108,NaN,1.0,NaN,NaN,NaN,87,19.310345
4,Pat:47:64938,St. Anton a. A. Terminal West,47.126527,10.263333,NaN,1.0,NaN,NaN,NaN,69,24.347826
...,...,...,...,...,...,...,...,...,...,...,...
2003,Pfl:21:810,Eschen Sportpark,47.205667,9.533982,NaN,1.0,NaN,NaN,NaN,186,9.032258
2004,Pfl:21:814,Eschen Kohlplatz,47.211068,9.528322,NaN,1.0,NaN,NaN,NaN,129,13.023256
2005,Pfl:21:841,Eschen Presta,47.207565,9.528179,NaN,1.0,NaN,NaN,NaN,216,7.777778
2006,Pfl:21:911,Schaan Theater,47.168200,9.512027,NaN,1.0,NaN,NaN,NaN,123,13.658537


In [ ]:
# Betrachtungszeitraum: 6–20 Uhr (= 840 Minuten)
# Stichtage: Werktag ohne Schule (Herbstferien): im Jahr 2021 der 28. 10.
# Normaler Werktag mit Schule: im Jahr 2021 der 22. 10.
# Intervallberechnung: Bildung der Summe der
# Abfahrten aller Verkehrsmittel über alle Ver-
# kehrsmittelkategorien, Multiplikation mit einem
# Richtungsfaktor von 0,5 und Berechnung des
# durchschnittlichen Intervalls über den gesamten
# Betrachtungszeitraum pro Richtung (840 Minuten
# dividiert durch die Zahl der Abfahrten pro Rich-
# tung). Der Richtungsfaktor wird auf allen Linien
# angewendet, Rundlinien ebenfalls.

# idea:
# 0. select specific day
# 1. take trip_id from entry in stop_times_df
# 2. look for that trip_id in trips_df
# 3. check if service_id from corresponding trip entry in calender_df is in valid time range,
#    also check exceptions in calender_date_df
# 4. if stop is valid, add to data for stop_id parent station
